## Notebook 02: ACS Demographic Data Preparation

### Purpose

This notebook downloads and processes Census ACS 5-year estimates at the census tract level for Kootenai County, Idaho.

The demographic variables include household characteristics, income, age, education, and housing tenure. These variables will be used to estimate spatial retail demand potential in later modeling steps.

### Data Source

- Source: U.S. Census Bureau American Community Survey (ACS) 5-Year Estimates
- Geography: Census tract
- Area: Kootenai County, Idaho
- Year: 2024

In [4]:
import pandas as pd
from pathlib import Path
import requests 

In [5]:
project_root = Path.cwd()
if not Path('data').exists(): 
    project_root = project_root.parent 
print(project_root)

d:\Wu\2026\Project Portfolio\003 Project\grocery-demand-estimation


### Downloading raw ACS variables

In [39]:
# constants
YEAR = '2024'
STATE = '16'
COUNTY = '055'
API_KEY = 'your_api_key'

In [ ]:
variables = [
    "NAME",         # tract name 
    "B11001_001E",  # household count
    "B25010_001E",  # average household size
    "B19013_001E",  # median household income
    "B01002_001E",  # median age
    "B25003_001E",  # occupied housing units
    "B25003_002E",  # owner occupied
    "B15003_001E",  # population 25+
    "B15003_022E",  # bachelor's
    "B15003_023E",  # master's
    "B15003_024E",  # professional
    "B15003_025E"   # doctorate
]

In [4]:
get_vars = ','.join(variables)
print(get_vars)

NAME,B11001_001E,B25010_001E,B19013_001E,B01002_001E,B25003_001E,B25003_002E,B15003_001E,B15003_022E,B15003_023E,B15003_024E,B15003_025E


#### Build the API URL dynamically

In [14]:
url = (
    f"https://api.census.gov/data/{YEAR}/acs/acs5"
    f"?get={get_vars}"
    "&for=tract:*"
    f"&in=state:{STATE}%20county:{COUNTY}"
    f"&key={API_KEY}"
)
print(url)

https://api.census.gov/data/2024/acs/acs5?get=NAME,B11001_001E,B25010_001E,B19013_001E,B01002_001E,B25003_001E,B25003_002E,B15003_001E,B15003_022E,B15003_023E,B15003_024E,B15003_025E&for=tract:*&in=state:16%20county:055&key=71183c4edbddf39da48305b7c7c09f2497c1892b


#### Send requests to Census API Server

In [15]:
response = requests.get(url)
print(response.status_code)
print(response.text[:300])

200
[["NAME","B11001_001E","B25010_001E","B19013_001E","B01002_001E","B25003_001E","B25003_002E","B15003_001E","B15003_022E","B15003_023E","B15003_024E","B15003_025E","state","county","tract"],
["Census Tract 1.01; Kootenai County; Idaho","1202","2.59","97381","51.3","1202","1058","2342","560","199","41


In [16]:
data = response.json()
type(data)

list

In [18]:
df = pd.DataFrame(data[1:], columns=data[0])
df.head()

,NAME,B11001_001E,B25010_001E,B19013_001E,B01002_001E,B25003_001E,B25003_002E,B15003_001E,B15003_022E,B15003_023E,B15003_024E,B15003_025E,state,county,tract
0,Census Tract 1.01; Kootenai County; Idaho,1202,2.59,97381,51.3,1202,1058,2342,560,199,41,38,16,055,000101
1,Census Tract 1.02; Kootenai County; Idaho,1847,2.56,96493,49.5,1847,1658,3586,619,191,0,26,16,055,000102
2,Census Tract 2.01; Kootenai County; Idaho,1330,2.92,109726,37.8,1330,1065,2776,561,123,14,36,16,055,000201
3,Census Tract 2.02; Kootenai County; Idaho,1312,2.70,92500,53.7,1312,1266,2615,436,275,72,20,16,055,000202
4,Census Tract 2.03; Kootenai County; Idaho,1407,2.41,70843,52.6,1407,1136,2538,620,56,0,0,16,055,000203


#### Create GEOID

In [20]:
df['GEOID'] = (
    df['state'].astype(str).str.zfill(2) +
    df['county'].astype(str).str.zfill(3) + 
    df['tract'].astype(str).str.zfill(6)
)

In [21]:
print(df.shape)
print(df.columns)

(39, 16)
Index(['NAME', 'B11001_001E', 'B25010_001E', 'B19013_001E', 'B01002_001E',
       'B25003_001E', 'B25003_002E', 'B15003_001E', 'B15003_022E',
       'B15003_023E', 'B15003_024E', 'B15003_025E', 'state', 'county', 'tract',
       'GEOID'],
      dtype='str')


#### Save raw ACS data

In [25]:
output_path = project_root / 'data/raw/kootenai_acs2024_raw.csv'

In [27]:
df.to_csv(output_path, index=False)
print(f"saved raw ACS data to {output_path}")

saved raw ACS data to d:\Wu\2026\Project Portfolio\003 Project\grocery-demand-estimation\data\raw\kootenai_acs2024_raw.csv


### Data Cleaning and Feature Creation

In [27]:
raw_path = project_root/'data/raw/kootenai_acs2024_raw.csv'
df = pd.read_csv(raw_path)

In [28]:
numeric_cols = [
    "B11001_001E",
    "B25010_001E",
    "B19013_001E",
    "B01002_001E",
    "B25003_001E",
    "B25003_002E",
    "B15003_001E",
    "B15003_022E",
    "B15003_023E",
    "B15003_024E",
    "B15003_025E"
]

df[numeric_cols] = df[numeric_cols].apply(
    pd.to_numeric, errors='coerce'
)

In [29]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   NAME         39 non-null     str    
 1   B11001_001E  39 non-null     int64  
 2   B25010_001E  39 non-null     float64
 3   B19013_001E  39 non-null     int64  
 4   B01002_001E  39 non-null     float64
 5   B25003_001E  39 non-null     int64  
 6   B25003_002E  39 non-null     int64  
 7   B15003_001E  39 non-null     int64  
 8   B15003_022E  39 non-null     int64  
 9   B15003_023E  39 non-null     int64  
 10  B15003_024E  39 non-null     int64  
 11  B15003_025E  39 non-null     int64  
 12  state        39 non-null     int64  
 13  county       39 non-null     int64  
 14  tract        39 non-null     int64  
 15  GEOID        39 non-null     int64  
dtypes: float64(2), int64(13), str(1)
memory usage: 5.0 KB


#### Rename variable into meaningful names

In [30]:
rename_dict = {
    "NAME": "tract_name",
    "B11001_001E": "household_count",
    "B25010_001E": "avg_household_size",
    "B19013_001E": "median_income",
    "B01002_001E": "median_age",
    "B25003_001E": "occupied_units",
    "B25003_002E": "owner_occupied",
    "B15003_001E": "population_25plus",
    "B15003_022E": "bachelors",
    "B15003_023E": "masters",
    "B15003_024E": "professional",
    "B15003_025E": "doctorate"
}

df = df.rename(columns=rename_dict)
df.columns

Index(['tract_name', 'household_count', 'avg_household_size', 'median_income',
       'median_age', 'occupied_units', 'owner_occupied', 'population_25plus',
       'bachelors', 'masters', 'professional', 'doctorate', 'state', 'county',
       'tract', 'GEOID'],
      dtype='str')

### Create derived variables

In [31]:
# owner rate 
df["owner_rate"] = (
    df["owner_occupied"] * 100
    / df["occupied_units"]
)

In [32]:
df["college_plus"] = (
    df["bachelors"]
    + df["masters"]
    + df["professional"]
    + df["doctorate"]
)

df["college_plus_pct"] = (
    df["college_plus"] * 100
    / df["population_25plus"]
)

In [33]:
df[
    [
        "GEOID",
        "owner_rate",
        "college_plus_pct"
    ]
].head()

,GEOID,owner_rate,college_plus_pct
0,16055000101,88.019967,35.781383
1,16055000102,89.767190,23.312883
2,16055000201,80.075188,26.440922
3,16055000202,96.493902,30.707457
4,16055000203,80.739161,26.635146


In [34]:
# remove unnecessary columns
df.columns


Index(['tract_name', 'household_count', 'avg_household_size', 'median_income',
       'median_age', 'occupied_units', 'owner_occupied', 'population_25plus',
       'bachelors', 'masters', 'professional', 'doctorate', 'state', 'county',
       'tract', 'GEOID', 'owner_rate', 'college_plus', 'college_plus_pct'],
      dtype='str')

In [35]:
final_cols = [
    'GEOID',
    'state', 
    'county',
    'tract',
    'household_count', 
    'avg_household_size', 
    'median_income',
    'median_age',
    'owner_rate',
    'college_plus_pct'
]

df_processed = df[final_cols].copy()

In [36]:
df_processed.head()

,GEOID,state,county,tract,household_count,avg_household_size,median_income,median_age,owner_rate,college_plus_pct
0,16055000101,16,55,101,1202,2.59,97381,51.3,88.019967,35.781383
1,16055000102,16,55,102,1847,2.56,96493,49.5,89.767190,23.312883
2,16055000201,16,55,201,1330,2.92,109726,37.8,80.075188,26.440922
3,16055000202,16,55,202,1312,2.70,92500,53.7,96.493902,30.707457
4,16055000203,16,55,203,1407,2.41,70843,52.6,80.739161,26.635146


In [38]:
processed_path = (
    project_root
    / "data"
    / "processed"
    / "kootenai_demographics2024.csv"
)

df_processed.to_csv(processed_path, index=False)
print(f"saved processed demographic data to {processed_path}")

saved processed demographic data to d:\Wu\2026\Project Portfolio\003 Project\grocery-demand-estimation\data\processed\kootenai_demographics2024.csv
